# 04 · GitHub、CI、第一个合并请求（路线第 1 步第 3 块）

**JD 对应**：「熟悉 Git/GitHub 协作流程（分支、Pull Request、code review）」「熟悉 CI/CD（GitHub Actions 等）」。

## 做了什么

```bash
gh repo create rare-disease-agent --private --source=. --remote=origin --push   # 建私有仓库，推上 main
git switch -c ci/github-actions                       # 开一个分支做这件事
# 写 .github/workflows/ci.yml；README 加 CI 徽章
git commit -m "ci: run the checks on GitHub for every pull request and push to main"
git push -u origin ci/github-actions                  # 推分支
gh pr create --base main --head ci/github-actions ... # 开第 1 个合并请求
```

然后把仓库的合并方式设成 **只允许 squash 合并，合并后自动删分支**（公司里最常见的设置）。

| 新东西 | 作用 |
|---|---|
| GitHub 上的私有仓库 `pipi0616/rare-disease-agent` | 正本从此在服务器上，本地只是工作副本 |
| `.github/workflows/ci.yml` | 每个合并请求、每次推到 main，GitHub 的机器自动跑 ruff、mypy、pytest |
| 合并请求 #1 | 第一次走完整流程：分支 → PR → 自动检查 → 合并 |
| README 徽章 | 一眼看到 CI 是绿是红 |

CI 配置文件只有 40 行，关键几处：

| 写法 | 意思 |
|---|---|
| `on: pull_request` + `push: branches: [main]` | 什么时候跑 |
| `permissions: contents: read` | 这个任务只读仓库，不给写权限（最小权限原则） |
| `concurrency: cancel-in-progress: true` | 同一分支又推了新的，就取消过时的那次 |
| `uv sync --locked` | 按采购单原样安装；采购单过期直接报错，依赖不能偷偷变 |
| `astral-sh/setup-uv@v10.2.0` | 精确版本号，不用会漂移的 `@v10` |

## 工业界叫什么

| 术语 | 大白话 |
|---|---|
| remote / origin | 服务器上的那份仓库；`origin` 是它的默认代号 |
| pull request（PR，合并请求） | "我在分支上改了这些，请审查并并入 main"。公司里所有改动都走这个 |
| CI（continuous integration，持续集成） | 每次改动自动在干净机器上跑全部检查。GitHub 自带的叫 GitHub Actions |
| workflow / job / step / runner | 一份 CI 配置 / 其中一个任务 / 任务里的一步 / 跑任务的那台机器 |
| status check（状态检查） | CI 结果挂在 PR 上的那个绿勾或红叉 |
| squash merge | 把分支上的多次提交压成一条并入 main；main 的历史一条 = 一个 PR |
| branch protection（分支保护） | 规定 main 只能通过 PR 修改、且检查必须绿。免费账号的私有仓库不开放，公开后再开 |

## 为什么

- **本地钩子是礼貌，CI 是闸门。** 钩子只在装了它的电脑上生效，克隆后忘了 `pre-commit install` 就没有了。
  CI 不管谁推的、从哪台机器推的，都在一台干净的 Ubuntu 上重跑一遍。
- **"在我电脑上能跑"不算数。** CI 的机器上什么都没有，只有仓库和采购单。它跑通了，才证明别人也能跑通。
- **结果挂在 PR 上。** 审查的人看绿勾就行，不用相信提交者说的"我测过了"。
- **最小权限、精确版本。** CI 配置是会被攻击的地方（供应链攻击）：少给权限、锁死版本，是公司安全审查的基本要求。

## CI 第一次是红的——怎么修的

第一次运行 3 秒就失败了。读日志（`gh run view <运行编号> --log-failed`），只有一行是重点：

```
##[error]Unable to resolve action `astral-sh/setup-uv@v10`, unable to find version `v10`
```

意思：这个 action 没有 `v10` 这个大版本标签。查它的标签列表（`gh api repos/astral-sh/setup-uv/tags`），
只有 `v10.2.0`、`v10.1.0` 这种精确版本。改成 `@v10.2.0`，再推一次，15 秒通过。

教训：**CI 红了先读日志，`##[error]` 那一行通常就是答案。** 以及：精确版本号本来就是更稳的写法。

## ✏️ 你来验证

下面的 `gh` 命令读的是 GitHub 上的真实状态。

In [1]:
import os, subprocess
ROOT = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(ROOT)
# notebook 里的 ! 命令跑在一个伪终端上，gh 会以为自己在真终端里，输出颜色码和转圈动画。关掉：
os.environ.update(TERM="dumb", NO_COLOR="1", GH_SPINNER_DISABLED="1", GH_PAGER="cat")
print("当前目录:", ROOT)

当前目录: /Users/pipi/Projects/QAFI_Paper/hearing_agent/rare-disease-agent


服务器上的仓库在哪、是否私有：

In [2]:
!git remote -v
!gh repo view --json nameWithOwner,visibility,url --jq '.nameWithOwner + "  " + .visibility + "  " + .url'

origin	https://github.com/pipi0616/rare-disease-agent.git (fetch)
origin	https://github.com/pipi0616/rare-disease-agent.git (push)


pipi0616/rare-disease-agent  PRIVATE  https://github.com/pipi0616/rare-disease-agent


第 1 个合并请求：

In [3]:
!gh pr view 1 --json number,title,state,headRefName,baseRefName,url --jq '"#" + (.number|tostring) + "  " + .title + "\n  " + .headRefName + " → " + .baseRefName + "  " + .state + "\n  " + .url'

Working...


#1  ci: run the checks on GitHub for every pull request
  ci/github-actions → main  OPEN
  https://github.com/pipi0616/rare-disease-agent/pull/1


挂在它上面的状态检查（应是 pass）：

In [4]:
!gh pr checks 1

Working...


All checks were successful
0 cancelled, 0 failing, 1 successful, 0 skipped, and 0 pending checks

   NAME                     DESCRIPTION  ELAPSED  URL                           
✓  CI/check (pull_request)               15s      https://github.com/pipi0616...


CI 的运行历史：第一次 failure，第二次 success：

In [5]:
!gh run list --branch ci/github-actions --limit 5 --json conclusion,createdAt,databaseId,headSha --jq '.[] | [.conclusion, .createdAt, (.databaseId|tostring), .headSha[:7]] | @tsv'

Working...


success	2026-10-07T09:08:26Z	37598537922	225dfd7
failure	2026-10-07T09:07:34Z	37598439620	b5062fc


仓库的合并设置：只允许 squash，合并后删分支：

In [6]:
!gh repo view --json squashMergeAllowed,mergeCommitAllowed,rebaseMergeAllowed,deleteBranchOnMerge

{
  "deleteBranchOnMerge": true,
  "mergeCommitAllowed": false,
  "rebaseMergeAllowed": false,
  "squashMergeAllowed": true
}


## 轮到你：点合并

这个 PR 由你来合并，这是以后每一步的固定动作：

1. 打开 PR 页面（上面 `gh pr view` 打印的网址）。
2. 看三样：改了哪些文件（Files changed）、CI 是否绿勾、提交信息说的和改的是否一致。
3. 点绿色按钮 **Squash and merge** → **Confirm**。分支会自动删除。

合并后在本地回到 main 并同步：

```bash
git switch main && git pull
```

## 思考

本地钩子已经全部通过了，为什么还要 CI 再跑一遍同样的检查？先想，再看下一格。

**答案**：三个原因。① 钩子只在装了它的电脑上有效，CI 对谁都有效。② CI 的机器上什么都没有，
跑通了才证明"不是只在我电脑上能跑"。③ CI 的结果挂在 PR 上，审查的人不用信任提交者的话。
公司里"CI 绿了才能合并"是硬规定，本地钩子只是让你少在 CI 上丢脸。